# Road Accident Severity Prediction — Feature Selection

This notebook follows the preprocessing principles used in the class notebooks:

- remove duplicates and the non-injury target class
- separate features and target
- create the train/test split before learning preprocessing statistics
- remove identifier and target-leakage columns
- calculate Information Gain on the training data only
- select the top informative original features
- save the selected feature list for the preprocessing stage

The feature-selection stage is kept separate from the final encoding/scaling stage.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

DATA_PATH = "../data/victorian_road_crash_data.csv"

df = pd.read_csv(DATA_PATH)

print("Original dataset shape:", df.shape)
display(df.head())


Original dataset shape: (200352, 52)


,ACCIDENT_NO,ACCIDENT_DATE,ACCIDENT_TIME,ACCIDENT_TYPE,DAY_OF_WEEK,DCA_CODE,DCA_CODE_DESCRIPTION,LIGHT_CONDITION,POLICE_ATTEND,ROAD_GEOMETRY,...,NO_OF_VEHICLES,HEAVYVEHICLE,PASSENGERVEHICLE,MOTORCYCLE,PT_VEHICLE,DEG_URBAN_NAME,SRNS,RMA,DIVIDED,STAT_DIV_NAME
0,T20140024624,27-11-2014,18:35:00,Collision with vehicle,Thursday,110,CROSS TRAFFIC(INTERSECTIONS ONLY),Day,Yes,Cross intersection,...,2.0,0.0,2.0,0.0,0.0,TOWNS,NaN,Local Road,Undivided,Country
1,T20190026336,27-12-2019,15:45:00,Collision with vehicle,Friday,113,RIGHT NEAR (INTERSECTIONS ONLY),Day,Yes,T intersection,...,2.0,0.0,2.0,0.0,0.0,MELB_URBAN,NaN,Arterial Other,Divided,Metro
2,T20190019196,02-10-2019,12:07:00,Collision with vehicle,Wednesday,173,RIGHT OFF CARRIAGEWAY INTO OBJECT/PARKED VEHICLE,Day,Yes,Not at intersection,...,3.0,0.0,2.0,0.0,0.0,MELB_URBAN,M,Freeway,Divided,Metro
3,T20250029202,07-11-2025,13:10:00,Struck Pedestrian,Friday,100,PED NEAR SIDE. PED HIT BY VEHICLE FROM THE RIGHT.,Day,Yes,Not at intersection,...,1.0,0.0,1.0,0.0,0.0,MELB_URBAN,NaN,Arterial Highway,Divided,Metro
4,T20210005363,30-01-2021,06:30:00,Collision with a fixed object,Saturday,183,OFF LEFT BEND INTO OBJECT/PARKED VEHICLE,Dusk/Dawn,No,Not at intersection,...,1.0,0.0,1.0,0.0,0.0,RURAL_VICTORIA,C,Arterial Other,Undivided,Metro


In [2]:
# Remove exact duplicate records

duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

df = df.drop_duplicates().copy()

print("Shape after removing duplicates:", df.shape)


Duplicate rows: 0
Shape after removing duplicates: (200352, 52)


In [3]:
# Remove the extremely small non-injury target class

print("Target distribution before removal:")
print(df["SEVERITY"].value_counts())

non_injury_count = (df["SEVERITY"] == "Non injury accident").sum()
print("\nNon-injury accidents:", non_injury_count)

df = df[df["SEVERITY"] != "Non injury accident"].copy()

print("\nTarget distribution after removal:")
print(df["SEVERITY"].value_counts())


Target distribution before removal:
SEVERITY
Other injury accident      124942
Serious injury accident     72054
Fatal accident               3352
Non injury accident             4
Name: count, dtype: int64

Non-injury accidents: 4

Target distribution after removal:
SEVERITY
Other injury accident      124942
Serious injury accident     72054
Fatal accident               3352
Name: count, dtype: int64


In [4]:
# Separate features and target

X = df.drop(columns=["SEVERITY"])
y = df["SEVERITY"]

print("X:", X.shape)
print("y:", y.shape)


X: (200348, 51)
y: (200348,)


In [5]:
# Train/test split BEFORE learning feature-selection mappings/statistics

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


X_train: (160278, 51)
X_test : (40070, 51)
y_train: (160278,)
y_test : (40070,)


## Remove identifier and target-leakage columns

These columns must not be allowed into feature selection or model training:

- `ACCIDENT_NO` — record identifier
- `INJ_OR_FATAL`
- `FATALITY`
- `SERIOUSINJURY`
- `OTHERINJURY`
- `NONINJURED`

The last five are derived from injury/severity information and would leak the target.


In [6]:
drop_columns = [
    "ACCIDENT_NO",
    "INJ_OR_FATAL",
    "FATALITY",
    "SERIOUSINJURY",
    "OTHERINJURY",
    "NONINJURED"
]

X_train = X_train.drop(columns=drop_columns)
X_test = X_test.drop(columns=drop_columns)

print("Removed columns:", drop_columns)
print("X_train after removal:", X_train.shape)
print("X_test after removal :", X_test.shape)


Removed columns: ['ACCIDENT_NO', 'INJ_OR_FATAL', 'FATALITY', 'SERIOUSINJURY', 'OTHERINJURY', 'NONINJURED']
X_train after removal: (160278, 45)
X_test after removal : (40070, 45)


## Information Gain

For Information Gain, categorical variables need a numeric representation.

Following the class approach, categorical values are filled using a training-set placeholder and label encoded. Numerical missing values are replaced using medians learned from the training set.

**Important:** the mappings and medians are learned from `X_train` only.


In [7]:
# Identify feature types from the training data

numeric_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Numerical features:", len(numeric_cols))
print("Categorical features:", len(categorical_cols))


Numerical features: 26
Categorical features: 19


In [8]:
# Build a training-only matrix for Information Gain

X_mi = X_train.copy()

# Numerical missing values: learn medians from training data only
if numeric_cols:
    numeric_imputer = SimpleImputer(strategy="median")
    X_mi[numeric_cols] = numeric_imputer.fit_transform(
        X_mi[numeric_cols]
    )

# Categorical missing values + label encoding
categorical_encoders = {}

for col in categorical_cols:
    X_mi[col] = X_mi[col].fillna("Missing").astype(str)

    encoder = LabelEncoder()
    X_mi[col] = encoder.fit_transform(X_mi[col])

    categorical_encoders[col] = encoder

# Safety handling for infinite values
X_mi = X_mi.replace([np.inf, -np.inf], np.nan)

if X_mi.isna().any().any():
    X_mi = X_mi.fillna(-1)

# Encode target using training labels only
target_encoder = LabelEncoder()
y_mi = target_encoder.fit_transform(y_train)

print("MI matrix shape:", X_mi.shape)
print("Missing values:", X_mi.isna().sum().sum())


MI matrix shape: (160278, 45)
Missing values: 0


In [9]:
# Calculate Information Gain / Mutual Information

mi_scores = mutual_info_classif(
    X_mi,
    y_mi,
    random_state=42
)

feature_scores = pd.DataFrame({
    "Feature": X_train.columns,
    "Information_Gain": mi_scores
}).sort_values(
    "Information_Gain",
    ascending=False
).reset_index(drop=True)

display(feature_scores)


,Feature,Information_Gain
0,POLICE_ATTEND,0.056059
1,DCA_CODE_DESCRIPTION,0.030759
2,DCA_CODE,0.029113
3,STAT_DIV_NAME,0.022607
4,NO_OF_VEHICLES,0.020218
5,DRIVER,0.018862
6,ACCIDENT_TYPE,0.018234
7,DEG_URBAN_NAME,0.018217
8,DIVIDED,0.017395
9,PASSENGERVEHICLE,0.017019


In [10]:
TOP_N = 20

selected_features = feature_scores.head(TOP_N)["Feature"].tolist()

print(f"Selected {len(selected_features)} features:")
for feature in selected_features:
    print("-", feature)


Selected 20 features:
- POLICE_ATTEND
- DCA_CODE_DESCRIPTION
- DCA_CODE
- STAT_DIV_NAME
- NO_OF_VEHICLES
- DRIVER
- ACCIDENT_TYPE
- DEG_URBAN_NAME
- DIVIDED
- PASSENGERVEHICLE
- ROAD_NAME
- LIGHT_CONDITION
- SPEED_ZONE
- LONGITUDE
- VICGRID_Y
- LATITUDE
- VICGRID_X
- ROAD_ROUTE_1
- SRNS
- MALES


In [11]:
# Check that no forbidden leakage/identifier column was selected

forbidden = set(drop_columns)
selected_forbidden = sorted(set(selected_features) & forbidden)

print("Forbidden columns selected:", selected_forbidden)
print("Zero-information features:",
      int((feature_scores["Information_Gain"] == 0).sum()))


Forbidden columns selected: []
Zero-information features: 0


In [12]:
# Save the selected feature names for the preprocessing notebook

selected_features_df = pd.DataFrame({
    "Feature": selected_features
})

selected_features_df.to_csv(
    "../data/processed/selected_features.csv",
    index=False
)

print("Saved: ../data/processed/selected_features.csv")


Saved: ../data/processed/selected_features.csv


## Final output

The next notebook should use `selected_features.csv` and apply the class preprocessing workflow:

**train/test split → missing-value handling → categorical encoding → target encoding → scaling → save processed data**
